# Phase 3 Feature Engineering

In [1]:
# ===========================================
# Import Required Libraries
# ===========================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

# Plot style
plt.style.use('ggplot')

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
# ===========================================
# Load Dataset
# ===========================================

file_path = "/content/drive/MyDrive/My Thesis/air_quality_cleaned.csv"

air_clean = pd.read_csv(file_path)

print("Dataset loaded successfully.")

Dataset loaded successfully.


In [4]:
air_clean.head()

,date,PM10,PM2.5,RH,Temp,WD,WS
0,2024-01-01 00:00:00,223.814714,65.119368,94.011898,28.618305,260.566102,0.604881
1,2024-01-01 01:00:00,343.645577,108.224682,91.278436,28.438167,279.908333,1.534783
2,2024-01-01 02:00:00,445.856723,140.734862,91.814506,27.912167,285.158333,1.327000
3,2024-01-01 03:00:00,407.405071,138.152391,90.311495,27.583500,289.420000,1.447833
4,2024-01-01 04:00:00,400.491289,126.808911,89.519261,27.357000,286.768333,1.326083


In [5]:
print(air_clean.shape)

(17544, 7)


In [6]:
# Make a copy to preserve the original cleaned dataset
air_feature = air_clean.copy()

### Checking the observation of the variables

In [7]:
air_feature.head()

,date,PM10,PM2.5,RH,Temp,WD,WS
0,2024-01-01 00:00:00,223.814714,65.119368,94.011898,28.618305,260.566102,0.604881
1,2024-01-01 01:00:00,343.645577,108.224682,91.278436,28.438167,279.908333,1.534783
2,2024-01-01 02:00:00,445.856723,140.734862,91.814506,27.912167,285.158333,1.327000
3,2024-01-01 03:00:00,407.405071,138.152391,90.311495,27.583500,289.420000,1.447833
4,2024-01-01 04:00:00,400.491289,126.808911,89.519261,27.357000,286.768333,1.326083


In [8]:
air_feature.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17544 entries, 0 to 17543
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   date    17544 non-null  object 
 1   PM10    17544 non-null  float64
 2   PM2.5   17544 non-null  float64
 3   RH      17544 non-null  float64
 4   Temp    17544 non-null  float64
 5   WD      17544 non-null  float64
 6   WS      15263 non-null  float64
dtypes: float64(6), object(1)
memory usage: 959.6+ KB


In [9]:
print(air_feature.index)

RangeIndex(start=0, stop=17544, step=1)


In [10]:
air_feature.columns

Index(['date', 'PM10', 'PM2.5', 'RH', 'Temp', 'WD', 'WS'], dtype='object')

In [11]:
# Conver the date to index
air_feature["date"] = pd.to_datetime(air_feature["date"])

air_feature = air_feature.set_index("date")

In [12]:
print(air_feature.index)

DatetimeIndex(['2024-01-01 00:00:00', '2024-01-01 01:00:00', '2024-01-01 02:00:00', '2024-01-01 03:00:00',
               '2024-01-01 04:00:00', '2024-01-01 05:00:00', '2024-01-01 06:00:00', '2024-01-01 07:00:00',
               '2024-01-01 08:00:00', '2024-01-01 09:00:00',
               ...
               '2025-12-31 14:00:00', '2025-12-31 15:00:00', '2025-12-31 16:00:00', '2025-12-31 17:00:00',
               '2025-12-31 18:00:00', '2025-12-31 19:00:00', '2025-12-31 20:00:00', '2025-12-31 21:00:00',
               '2025-12-31 22:00:00', '2025-12-31 23:00:00'],
              dtype='datetime64[ns]', name='date', length=17544, freq=None)


In [13]:
# Hour Feature
air_feature["Hour"] = air_feature.index.hour

In [14]:
# Dau of the Week
# Monday = 0, Sunday = 6
air_feature["DayOfWeek"] = air_feature.index.dayofweek

In [15]:
# Weekend Indicator
air_feature["Weekend"] = (
    air_feature["DayOfWeek"] >= 5
).astype(int)

In [16]:
# Day of the Month
air_feature["Day"] = air_feature.index.day

In [17]:
# Month
air_feature["Month"] = air_feature.index.month

In [18]:
# Quater of the Year
air_feature["Quarter"] = air_feature.index.quarter

In [19]:
# Year
air_feature["Year"] = air_feature.index.year

In [20]:
# Day of the Year
air_feature["DayOfYear"] = air_feature.index.dayofyear

In [21]:
# Ghanaian climatic seasons
def ghana_season(month):

    if month in [11, 12, 1, 2]:
        return "Dry_Season"

    elif month in [3, 4, 5]:
        return "Minor_Rainy_Season"

    elif month in [6, 7, 8]:
        return "Major_Rainy_Season"

    else:   # September, October
        return "Minor_Rainy_Season"

air_feature["Season"] = air_feature["Month"].apply(ghana_season)

In [22]:
# Very the Season
air_feature["Season"].value_counts()

,count
Season,
Minor_Rainy_Season,7344
Dry_Season,5784
Major_Rainy_Season,4416


In [23]:
# Harmattan indicator
air_feature["Harmattan"] = (
    air_feature["Month"]
    .isin([11, 12, 1, 2])
    .astype(int)
)

air_feature["Harmattan"].value_counts()

,count
Harmattan,
0,11760
1,5784


In [24]:
air_feature[
    [
        "Hour",
        "DayOfWeek",
        "Weekend",
        "Day",
        "Month",
        "Quarter",
        "Year",
        "DayOfYear",
        "Season",
        "Harmattan"
    ]
].head(10)

,Hour,DayOfWeek,Weekend,Day,Month,Quarter,Year,DayOfYear,Season,Harmattan
date,,,,,,,,,,
2024-01-01 00:00:00,0,0,0,1,1,1,2024,1,Dry_Season,1
2024-01-01 01:00:00,1,0,0,1,1,1,2024,1,Dry_Season,1
2024-01-01 02:00:00,2,0,0,1,1,1,2024,1,Dry_Season,1
2024-01-01 03:00:00,3,0,0,1,1,1,2024,1,Dry_Season,1
2024-01-01 04:00:00,4,0,0,1,1,1,2024,1,Dry_Season,1
2024-01-01 05:00:00,5,0,0,1,1,1,2024,1,Dry_Season,1
2024-01-01 06:00:00,6,0,0,1,1,1,2024,1,Dry_Season,1
2024-01-01 07:00:00,7,0,0,1,1,1,2024,1,Dry_Season,1
2024-01-01 08:00:00,8,0,0,1,1,1,2024,1,Dry_Season,1


In [25]:
air_feature[
    [
        "Hour",
        "DayOfWeek",
        "Weekend",
        "Day",
        "Month",
        "Quarter",
        "Year",
        "DayOfYear",
        "Season",
        "Harmattan"
    ]
].isna().sum()

,0
Hour,0
DayOfWeek,0
Weekend,0
Day,0
Month,0
Quarter,0
Year,0
DayOfYear,0
Season,0
Harmattan,0


# Cyclical feature Engineering

In [26]:
# Hour encoding
air_feature["Hour_sin"] = np.sin(
    2 * np.pi * air_feature["Hour"] / 24
)

air_feature["Hour_cos"] = np.cos(
    2 * np.pi * air_feature["Hour"] / 24
)

In [27]:
# Wind direction encoding
air_feature["WD_sin"] = np.sin(
    2 * np.pi * air_feature["WD"] / 360
)

air_feature["WD_cos"] = np.cos(
    2 * np.pi * air_feature["WD"] / 360
)

In [28]:
[col for col in air_feature.columns if "sin" in col or "cos" in col]

['Hour_sin', 'Hour_cos', 'WD_sin', 'WD_cos']

In [29]:
# Month cyclical encoding
air_feature["Month_sin"] = np.sin(
    2 * np.pi * air_feature["Month"] / 12
)

air_feature["Month_cos"] = np.cos(
    2 * np.pi * air_feature["Month"] / 12
)

In [30]:
air_feature[
    [
        "Hour",
        "Hour_sin",
        "Hour_cos",
        "Month",
        "Month_sin",
        "Month_cos",
        "WD",
        "WD_sin",
        "WD_cos"
    ]
].head()

,Hour,Hour_sin,Hour_cos,Month,Month_sin,Month_cos,WD,WD_sin,WD_cos
date,,,,,,,,,
2024-01-01 00:00:00,0,0.000000,1.000000,1,0.5,0.866025,260.566102,-0.986475,-0.163910
2024-01-01 01:00:00,1,0.258819,0.965926,1,0.5,0.866025,279.908333,-0.985084,0.172072
2024-01-01 02:00:00,2,0.500000,0.866025,1,0.5,0.866025,285.158333,-0.965207,0.261487
2024-01-01 03:00:00,3,0.707107,0.707107,1,0.5,0.866025,289.420000,-0.943107,0.332490
2024-01-01 04:00:00,4,0.866025,0.500000,1,0.5,0.866025,286.768333,-0.957479,0.288503


In [31]:
cyclical_features = [
    "Hour_sin",
    "Hour_cos",
    "Month_sin",
    "Month_cos",
    "WD_sin",
    "WD_cos"
]

print(air_feature[cyclical_features].head())

                     Hour_sin  Hour_cos  Month_sin  Month_cos    WD_sin    WD_cos
date                                                                             
2024-01-01 00:00:00  0.000000  1.000000        0.5   0.866025 -0.986475 -0.163910
2024-01-01 01:00:00  0.258819  0.965926        0.5   0.866025 -0.985084  0.172072
2024-01-01 02:00:00  0.500000  0.866025        0.5   0.866025 -0.965207  0.261487
2024-01-01 03:00:00  0.707107  0.707107        0.5   0.866025 -0.943107  0.332490
2024-01-01 04:00:00  0.866025  0.500000        0.5   0.866025 -0.957479  0.288503


### Create lag features

In [32]:
# PM2.5 lag features
air_feature["PM2.5_lag1"] = air_feature["PM2.5"].shift(1)

air_feature["PM2.5_lag3"] = air_feature["PM2.5"].shift(3)

air_feature["PM2.5_lag6"] = air_feature["PM2.5"].shift(6)

air_feature["PM2.5_lag12"] = air_feature["PM2.5"].shift(12)

air_feature["PM2.5_lag24"] = air_feature["PM2.5"].shift(24)

In [33]:
air_feature[
    [
        "PM2.5",
        "PM2.5_lag1",
        "PM2.5_lag3",
        "PM2.5_lag6",
        "PM2.5_lag12",
        "PM2.5_lag24"
    ]
].head(30)

,PM2.5,PM2.5_lag1,PM2.5_lag3,PM2.5_lag6,PM2.5_lag12,PM2.5_lag24
date,,,,,,
2024-01-01 00:00:00,65.119368,NaN,NaN,NaN,NaN,NaN
2024-01-01 01:00:00,108.224682,65.119368,NaN,NaN,NaN,NaN
2024-01-01 02:00:00,140.734862,108.224682,NaN,NaN,NaN,NaN
2024-01-01 03:00:00,138.152391,140.734862,65.119368,NaN,NaN,NaN
2024-01-01 04:00:00,126.808911,138.152391,108.224682,NaN,NaN,NaN
2024-01-01 05:00:00,92.292091,126.808911,140.734862,NaN,NaN,NaN
2024-01-01 06:00:00,89.009087,92.292091,138.152391,65.119368,NaN,NaN
2024-01-01 07:00:00,85.023910,89.009087,126.808911,108.224682,NaN,NaN
2024-01-01 08:00:00,73.683788,85.023910,92.292091,140.734862,NaN,NaN


In [34]:
# Verify Missing values
air_feature[
    [
        "PM2.5_lag1",
        "PM2.5_lag3",
        "PM2.5_lag6",
        "PM2.5_lag12",
        "PM2.5_lag24"
    ]
].isna().sum()

,0
PM2.5_lag1,1
PM2.5_lag3,3
PM2.5_lag6,6
PM2.5_lag12,12
PM2.5_lag24,24


In [35]:
# air_model = air_feature.dropna().copy()

# Rolling Statistical Features

In [39]:
# Previous 6-hour average
air_feature["PM2.5_rollmean6"] = (
    air_feature["PM2.5"]
    .rolling(window=6)
    .mean()
)

# Previous 12-hour average
air_feature["PM2.5_rollmean12"] = (
    air_feature["PM2.5"]
    .rolling(window=12)
    .mean()
)


# Previous 12-hour average
air_feature["PM2.5_rollmean12"] = (
    air_feature["PM2.5"]
    .rolling(window=12)
    .mean()
)


# Previous 24-hour average
air_feature["PM2.5_rollmean24"] = (
    air_feature["PM2.5"]
    .rolling(window=24)
    .mean()
)

# Previous 24-hour variability
air_feature["PM2.5_rollstd24"] = (
    air_feature["PM2.5"]
    .rolling(window=24)
    .std()
)

In [40]:
# Verify the features
air_feature[
    [
        "PM2.5",
        "PM2.5_rollmean6",
        "PM2.5_rollmean12",
        "PM2.5_rollmean24",
        "PM2.5_rollstd24"
    ]
].head(30)

,PM2.5,PM2.5_rollmean6,PM2.5_rollmean12,PM2.5_rollmean24,PM2.5_rollstd24
date,,,,,
2024-01-01 00:00:00,65.119368,NaN,NaN,NaN,NaN
2024-01-01 01:00:00,108.224682,NaN,NaN,NaN,NaN
2024-01-01 02:00:00,140.734862,NaN,NaN,NaN,NaN
2024-01-01 03:00:00,138.152391,NaN,NaN,NaN,NaN
2024-01-01 04:00:00,126.808911,NaN,NaN,NaN,NaN
2024-01-01 05:00:00,92.292091,111.888718,NaN,NaN,NaN
2024-01-01 06:00:00,89.009087,115.870337,NaN,NaN,NaN
2024-01-01 07:00:00,85.023910,112.003542,NaN,NaN,NaN
2024-01-01 08:00:00,73.683788,100.828363,NaN,NaN,NaN


In [41]:
air_feature[
    [
        "PM2.5_rollmean6",
        "PM2.5_rollmean12",
        "PM2.5_rollmean24",
        "PM2.5_rollstd24"
    ]
].isna().sum()

,0
PM2.5_rollmean6,5
PM2.5_rollmean12,11
PM2.5_rollmean24,23
PM2.5_rollstd24,23


In [42]:
# View the Summary
air_feature[
    [
        "PM2.5_rollmean6",
        "PM2.5_rollmean12",
        "PM2.5_rollmean24",
        "PM2.5_rollstd24"
    ]
].describe()

,PM2.5_rollmean6,PM2.5_rollmean12,PM2.5_rollmean24,PM2.5_rollstd24
count,17539.000000,17533.000000,17521.000000,17521.000000
mean,24.707296,24.691613,24.675567,5.639119
std,26.685061,26.375330,25.992272,5.123488
min,2.305062,3.214101,5.580117,0.282812
25%,12.698178,13.027860,13.297308,2.926203
50%,17.321451,17.233260,17.158456,4.332274
75%,26.857883,26.563493,26.480885,6.630633
max,247.621201,240.169688,228.560859,83.077475


# Meteorological Interaction Features

In [44]:
# Temperature × Relative Humidity
air_feature["Temp_RH"] = (
    air_feature["Temp"] *
    air_feature["RH"]
)

# PM2.5 × Relative Humidity
air_feature["PM2.5_RH"] = (
    air_feature["PM2.5"] *
    air_feature["RH"]
)

# PM2.5 & PM10 ratio
air_feature["PM2.5_PM10_ratio"] = (
    air_feature["PM2.5"] /
    air_feature["PM10"].replace(0, np.nan)
)

In [45]:
# Verify the Features
air_feature[
[
    "Temp_RH",
    "PM2.5_RH",
    "PM2.5_PM10_ratio"
]
].describe()

,Temp_RH,PM2.5_RH,PM2.5_PM10_ratio
count,17544.000000,17544.000000,17544.000000
mean,2545.777143,2148.195663,0.386778
std,210.919123,1973.571910,0.093048
min,885.439943,149.608047,0.041105
25%,2442.760680,1128.899206,0.320201
50%,2551.887281,1602.161678,0.371650
75%,2678.332290,2450.174320,0.446587
max,3078.125671,23054.160261,0.778177


In [46]:
air_feature[
[
    "Temp_RH",
    "PM2.5_RH",
    "PM2.5_PM10_ratio"
]
].isna().sum()

,0
Temp_RH,0
PM2.5_RH,0
PM2.5_PM10_ratio,0


In [47]:
# Do the ratio during Harmattan?
air_feature.groupby("Harmattan")[
    "PM2.5_PM10_ratio"
].describe()

,count,mean,std,min,25%,50%,75%,max
Harmattan,,,,,,,,
0,11760.0,0.409921,0.094568,0.041498,0.337376,0.397393,0.477342,0.778177
1,5784.0,0.339723,0.069105,0.041105,0.293959,0.335899,0.382652,0.670459


In [48]:
air_feature.groupby("Harmattan")[["PM2.5","PM10"]].mean()

,PM2.5,PM10
Harmattan,,
0,15.442387,39.845073
1,43.596636,149.593084


In [49]:
# Save the data Engineered dataset
air_feature.to_csv(
    "air_quality_feature_engineered.csv",
    index=True
)